In [2]:
!pip install -qU transformers accelerate bitsandbytes huggingface_hub

In [3]:
import torch
print("GPU Available:", torch.cuda.is_available())

GPU Available: True


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
print("Model loaded successfully!")

config.json:   0%|          | 0.00/680 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.07k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/28.1k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
# Math test
messages = [
    {"role": "user", "content": "A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost? Show your reasoning."}
]

# 1. Apply chat template → returns BatchEncoding with input_ids, attention_mask
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,  # explicit
).to("cuda")

# 2. Generate — pass the whole BatchEncoding (has input_ids + attention_mask)
outputs = model.generate(
    **inputs,                    # unpacks input_ids, attention_mask
    max_new_tokens=1024,
    temperature=0.6,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
)

# 3. Decode only new tokens
input_len = inputs["input_ids"].shape[-1]   # or inputs.input_ids.shape[-1]
response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
print(response)


Okay, so I've got this problem here: "A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost? Show your reasoning." Hmm, at first glance, it seems pretty straightforward, but I remember hearing that this problem is a classic example of something that seems simple but can be tricky if you don't think carefully. Let me try to break it down step by step.

First, let's define what we're trying to find. We need to find the cost of the ball. Let's denote the cost of the ball as "x" dollars. That makes sense, right? So if the ball is x dollars, then the bat must be x plus $1.00, because it's $1.00 more than the ball.

So, mathematically, that would be:
- Ball = x
- Bat = x + 1.00

Together, they cost $1.10. So, if I add the cost of the ball and the cost of the bat, it should equal $1.10. Putting that into an equation, it would be:

x (ball) + (x + 1.00) (bat) = 1.10

Simplifying that equation, I can combine like terms. So, x + x is 2x, and th

In [ ]:
import re
import torch

def chat(messages, max_new_tokens=512, temperature=0.7, top_p=0.9):
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.05,
        )

    input_len = inputs["input_ids"].shape[-1]
    raw_response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

    # Remove the <think>...</think> block (including multiline contents)
    cleaned_response = re.sub(r'<think>.*?</think>', '', raw_response, flags=re.DOTALL).strip()

    return cleaned_response

In [ ]:
# Test 2
print(chat([{"role": "user", "content": "Write a Python LRU cache."}]))


Okay, I need to write a Python LRU cache. Let me think about how to approach this.

First, I know that Python has some built-in modules like `functools` which have `lru_cache`, but I guess the task is to create a simple version of that myself.

So, what's an LRU cache? It's a data structure that keeps track of the most recently used items. When the cache is full and a new item is added, the least recently used item gets evicted.

I need to implement something similar. The basic operations would be `get` and `put`.

Let me outline the steps:

1. **Initialization**: The cache should have a fixed size. I'll need to initialize it with some maximum size, say 256 as a default. Also, I'll need counters to track the usage order of each item.

2. **Tracking Usage**: Each time an item is accessed (either read or modified), its usage timestamp should be updated. This helps determine the least recently used item when a eviction is needed.

3. **Eviction Strategy**: When the cache reaches its capac

In [ ]:
# Multi-turn conversation
messages = [
    {"role": "user", "content": "Explain transformers in 2 sentences."},
    {"role": "assistant", "content": "Transformers use self-attention to process sequences in parallel, capturing relationships between all tokens simultaneously."},
    {"role": "user", "content": "Now explain it like I'm 5."},
]
print(chat(messages))


Okay, so the user first asked me to explain transformers in two sentences. They probably want a basic understanding without too much jargon. Then they followed up by asking me to explain it like I'm five. That means they're really looking for something even simpler and more relatable.

I need to break down what transformers are into something a child can grasp. Maybe using a familiar analogy, like a library or a bookshelf, would help. I should focus on the main idea: moving through the books quickly instead of one by one. This shows efficiency, which is key.

I'll compare the layers to workers in a factory. Each layer does a small part, but together they handle everything. This makes it clear how transformers work step by step without getting too complicated.

Finally, I'll mention how transformers can learn from data and recognize patterns. Keeping it positive and encouraging will make the explanation feel helpful and approachable.
</think>

Imagine you have a big stack of books, and 

In [ ]:
# Test 4
print(chat([{"role": "user", "content": "Write a reusable react button component with register test, styled with tailwind css v4"}]))


Okay, I need to create a reusable React button component that uses Tailwind CSS v4 and includes unit tests. Let me break this down.

First, the component should be reusable, so I'll structure it as a functional component in a .tsx file. I'll import React and any necessary hooks, like useState.

Next, styling with Tailwind CSS v4. I remember that v4 introduced utility classes for components, but for styling, I'll use the newer utility classes for colors, spacing, etc. I'll choose a color scheme—maybe a primary color for the button, with hover and active states.

The component needs to have text, so I'll include an input field and a button. I'll manage the input's state with useState. When the button is clicked, it should add the text to a list and clear the input. That means I'll need a state variable to keep track of the added items.

For the tests, I'll use Jest since it's commonly used for React testing. I'll create integration tests because the component is part of a larger applicat

In [ ]:
# Test 5
print(chat([{"role": "user", "content": "What is the differentiation of x^3 - 3x^2 + 5"}]))


Alright, so I have to find the derivative of this function: x cubed minus three x squared plus five. Hmm, okay, let me think about how to approach this. I remember from my math class that when you take derivatives, you can do it term by term. So, I'll break it down into each part and then combine the results.

First, let's look at the first term, which is x cubed. I think the rule for differentiating something like x raised to a power is to bring down the exponent as a coefficient, then subtract one from the exponent. So, for x cubed, the exponent is 3. Bringing the 3 down as a coefficient, I get 3 times x to the power of (3 minus 1), which is 2. So that gives me 3x squared. Okay, that seems straightforward.

Next, the second term is negative three x squared. Let me handle the negative sign first because it might make things clearer. So, it's minus 3 times x squared. Now, applying the same rule as before, the exponent on x is 2. Bringing the 2 down as a coefficient, I get 2 times 3, wh

In [ ]:
# Test 5
print(chat([{"role": "user", "content": "Write a cover letter for a junior AI engineer with a bachelor's degree in CS"}]))

Okay, I need to write a cover letter for a junior AI engineer position with a bachelor's degree in Computer Science. Let me start by understanding what the user is asking for. They provided a sample cover letter, so I should use that as a reference but also think about how to make it unique and tailored.

First, I'll consider the structure of the cover letter. It should have the sender's contact information at the top, followed by the date and recipient's address. Then a greeting, the body, and a closing with contact info again.

In the body, I need to introduce myself, mention my education, and then talk about my experience in AI and machine learning. I should highlight specific skills like programming languages (Python, TensorFlow, PyTorch), machine learning algorithms, deep learning, and experience with NLP or computer vision.

I should also include something about problem-solving, maybe a project I worked on, like building a recommendation system or implementing a language model. T

In [ ]:
# Test 6
print(chat([{"role": "user", "content": "Write a function in Javascript"}]))


Okay, the user asked me to write a function in JavaScript. Let me think about what they might need. They didn't specify the purpose, so I should consider common functions people use.

Maybe they want a greeting function. That's simple and helpful. So I'll create a function called greet that takes no arguments and returns a friendly message.

I should make sure it's clear and easy to understand. Using console.log will show the output, which is good for testing.

I wonder if they know how to define functions. I'll structure it properly with function name and parameters, then return the string.

Perhaps they're just starting out, so keeping it simple is best. I'll provide the code and explain each part so they can see how it works.

I should also mention how they can call the function later, like greet().

Overall, I want to make sure they can easily follow along and implement the function without confusion.
</think>

Here's an example of a simple JavaScript function that returns a greeti

In [ ]:
import json, time, torch, re
from tabulate import tabulate

# ─── Test Suite: All 7 Dimensions ───────────────────────────────
TESTS = [
    # Reasoning & Logic
    {"dim": "Reasoning", "name": "Train Problem", "prompt":
     "A train leaves A at 60mph, another leaves B at 80mph toward A. Distance 420mi. When do they meet? Show steps.",
     "check": lambda r: "3" in r and ("hour" in r.lower() or "h" in r.lower())},

    {"dim": "Reasoning", "name": "Bat & Ball", "prompt":
     "A bat and a ball cost $1.10 total. The bat costs $1.00 more than the ball. How much is the ball? Show reasoning.",
     "check": lambda r: "0.05" in r or "5 cent" in r.lower()},

    # Coding
    {"dim": "Coding", "name": "LRU Cache", "prompt":
     "Write a Python LRU cache class with capacity N, get/put O(1), type hints, docstring.",
     "check": lambda r: "class" in r and ("OrderedDict" in r or "dict" in r) and "def get" in r and "def put" in r},

    {"dim": "Coding", "name": "Bug Fix", "prompt":
     "Fix the bug:\ndef find_max(arr):\n    max_val = 0\n    for i in arr:\n        if i > max_val:\n            max_val = i\n    return max_val",
     "check": lambda r: "max_val = arr[0]" in r or "float('-inf')" in r or "if not arr" in r},

    {"dim": "Coding", "name": "Regex Explain", "prompt":
     "Explain this regex: r'(?<!\\\\)\\\\.(?=([^\"\\\\]*(\\\\.|\\\"([^\"\\\\]*\\\\.)*[^\"\\\\]*\\\"))*[^\"\\\\]*$)'",
     "check": lambda r: "escape" in r.lower() or "dot" in r.lower() or "quote" in r.lower()},

    # Instruction Following
    {"dim": "Instruction Following", "name": "JSON Only", "prompt":
     "Return ONLY valid JSON with keys: name, email, risk_level. No markdown. Data: John Doe, john@co.com, high",
     "check": lambda r: r.strip().startswith("{") and r.strip().endswith("}") and "name" in r and "email" in r},

    # Writing & Communication
    {"dim": "Writing", "name": "CEO Summary", "prompt":
     "Summarize in 3 bullets for a CEO: 'Transformer models use self-attention to process sequences in parallel. Unlike RNNs, they don't process tokens sequentially, enabling GPU utilization. The attention mechanism computes relationships between all token pairs. Multi-head attention allows capturing different relationship types. Positional encodings inject order information. Layer normalization and residual connections stabilize deep training. Feed-forward networks apply pointwise transformations. This architecture scales effectively to billions of parameters.'",
     "check": lambda r: r.count("\n") >= 2 or r.count("•") >= 2 or r.count("-") >= 2},

    {"dim": "Writing", "name": "Delay Email", "prompt":
     "Write a professional email to a client explaining a 2-week project delay due to unexpected technical debt, proposing a revised timeline and mitigation steps.",
     "check": lambda r: "delay" in r.lower() and ("technical debt" in r.lower() or "revised" in r.lower())},

    # Structured Output
    {"dim": "Structured Output", "name": "Entity Extraction", "prompt":
     'Extract name, email, company, pain_points as JSON: "Hi, I\'m Sarah Chen (sarah@techcorp.io) from TechCorp. We struggle with slow CI/CD and flaky tests."',
     "check": lambda r: "sarah" in r.lower() and "techcorp" in r.lower() and "ci/cd" in r.lower()},

    {"dim": "Structured Output", "name": "NL to SQL", "prompt":
     "Generate SQL: 'Show all users who signed up in January 2024 and have made at least 3 orders.' Tables: users(id, created_at), orders(id, user_id, created_at)",
     "check": lambda r: "SELECT" in r.upper() and "JOIN" in r.upper() and "COUNT" in r.upper()},

    # Context Utilization
    {"dim": "Context", "name": "Long Context QA", "prompt":
     "Context: " + "The project codename is 'Project Falcon'. The budget is $2.4M. The deadline is Q3 2025. " * 200 +
     "\n\nQuestion: What is the project codename and budget? Answer based ONLY on the text above.",
     "check": lambda r: "falcon" in r.lower() and "2.4" in r},

    # Multilingual
    {"dim": "Multilingual", "name": "Japanese Translation", "prompt":
     "Translate to Japanese: 'Our Q3 revenue increased 12% YoY driven by APAC expansion.'",
     "check": lambda r: any(c in r for c in "増収益四半期")},
]

# ─── Helpers ────────────────────────────────────────────────────
def chat(messages, max_new_tokens=512, temperature=0.3, top_p=0.9):
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, temperature=temperature,
            top_p=top_p, do_sample=True, pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.05,
        )
    input_len = inputs["input_ids"].shape[-1]
    return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

def clean_response(text: str) -> str:
    """Strip DeepSeek-R1 thinking tags using regex"""
    return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()

def benchmark(prompt, max_new=512, runs=3):
    """Speed + PyTorch VRAM measurement"""
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)

    # Warm-up
    _ = model.generate(**inputs, max_new_tokens=16)
    torch.cuda.synchronize()

    times, tokens = [], []
    for _ in range(runs):
        start = time.time()
        out = model.generate(**inputs, max_new_tokens=max_new, temperature=0.7, do_sample=True)
        torch.cuda.synchronize()
        elapsed = time.time() - start
        new_toks = out.shape[1] - inputs["input_ids"].shape[1]
        times.append(elapsed)
        tokens.append(new_toks)

    avg_toks = sum(tokens)/len(tokens)
    avg_time = sum(times)/len(times)

    # Get VRAM usage directly from PyTorch (in MB)
    vram_mb = torch.cuda.memory_allocated() / (1024 ** 2)

    return {
        "tok_s": round(avg_toks/avg_time, 1),
        "latency_s": round(avg_time, 2),
        "vram_mb": round(vram_mb, 1),
    }

# ─── Run All Tests ──────────────────────────────────────────────
results = []
print(f"{'='*60}\nRUNNING {len(TESTS)} TESTS\n{'='*60}")

for i, test in enumerate(TESTS, 1):
    print(f"\n[{i}/{len(TESTS)}] {test['dim']} - {test['name']}")

    # Generate
    raw = chat([{"role": "user", "content": test["prompt"]}])
    cleaned = clean_response(raw)

    # Auto-score
    passed = test["check"](cleaned)
    score = 1 if passed else 0

    # Benchmark
    perf = benchmark(test["prompt"])

    results.append({
        "dimension": test["dim"],
        "test": test["name"],
        "passed": passed,
        "score": score,
        "tok_s": perf["tok_s"],
        "latency_s": perf["latency_s"],
        "vram_mb": perf["vram_mb"],
        "response": cleaned[:300] + "..." if len(cleaned) > 300 else cleaned,
    })

    status = "✅ PASS" if passed else "❌ FAIL"
    print(f"  {status} | {perf['tok_s']} tok/s | {perf['latency_s']}s | VRAM: {perf['vram_mb']}MB")
    print(f"  Response: {cleaned[:150]}...")

# ─── Summary Tables ─────────────────────────────────────────────
print(f"\n{'='*60}\nDIMENSION SUMMARY\n{'='*60}")

dim_scores = {}
for r in results:
    dim_scores.setdefault(r["dimension"], {"pass": 0, "total": 0, "tok_s": [], "latency": []})
    dim_scores[r["dimension"]]["pass"] += r["score"]
    dim_scores[r["dimension"]]["total"] += 1
    dim_scores[r["dimension"]]["tok_s"].append(r["tok_s"])
    dim_scores[r["dimension"]]["latency"].append(r["latency_s"])

summary_rows = []
for dim, stats in dim_scores.items():
    pct = stats["pass"] / stats["total"] * 100
    avg_tok = sum(stats["tok_s"])/len(stats["tok_s"])
    avg_lat = sum(stats["latency"])/len(stats["latency"])
    summary_rows.append([dim, f"{stats['pass']}/{stats['total']}", f"{pct:.0f}%", f"{avg_tok:.1f}", f"{avg_lat:.2f}s"])

print(tabulate(summary_rows, headers=["Dimension", "Passed", "Score", "Avg tok/s", "Avg latency"], tablefmt="github"))

# Overall (Fixed syntax error)
total_pass = sum(r["score"] for r in results)
total_all = len(results)
print(f"\nOVERALL: {total_pass}/{total_all} ({total_pass/total_all*100:.0f}%)")

# ─── Save Full Results ──────────────────────────────────────────
with open("/content/capability_eval_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("\n✅ Full results saved to /content/capability_eval_results.json")

# ─── Per-Test Detail (for manual review) ────────────────────────
print(f"\n{'='*60}\nPER-TEST DETAIL\n{'='*60}")
detail_rows = [[r["dimension"], r["test"], "✅" if r["passed"] else "❌", f"{r['tok_s']:.1f}", r["response"][:100]] for r in results]
print(tabulate(detail_rows, headers=["Dim", "Test", "Pass", "tok/s", "Response (truncated)"], tablefmt="github"))

RUNNING 12 TESTS

[1/12] Reasoning - Train Problem
  ✅ PASS | 14.4 tok/s | 32.48s | VRAM: 5594.1MB
  Response: First, I need to determine the total distance between points A and B, which is 420 miles.

Next, I'll identify the speeds of both trains. The train le...

[2/12] Reasoning - Bat & Ball
  ✅ PASS | 14.6 tok/s | 34.49s | VRAM: 5594.1MB
  Response: Okay, so I have this problem here: A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much is the ball? Hmm, let me try...

[3/12] Coding - LRU Cache
  ❌ FAIL | 14.6 tok/s | 35.04s | VRAM: 5594.1MB
  Response: Okay, I need to write a Python LRU cache class with capacity N, and it should have get and put methods with O(1) time complexity. Also, it should incl...

[4/12] Coding - Bug Fix
  ❌ FAIL | 14.6 tok/s | 35.16s | VRAM: 5594.1MB
  Response: Okay, so I have this Python function called find_max that's supposed to find the maximum value in an array, but it's buggy. Let me see what's wrong wi...

[5/12] Co

KeyboardInterrupt: 